## let's convert columns to the right type

In [55]:
import pandas as pd
import numpy as np
import warnings

# Ignore future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [56]:
# -------------------------------
# Step 1: Load the Dataset
# -------------------------------
df = pd.read_csv('D:\Data science\Qasim Elasyed Gadelkareem Ali\Csv Files\insurance_dirty.csv')

print(f"✅ Dataset loaded. Shape: {df.shape}")

<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Qasim\AppData\Local\Temp\ipykernel_25412\2820980501.py:4: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('D:\Data science\Qasim Elasyed Gadelkareem Ali\Csv Files\insurance_dirty.csv')


✅ Dataset loaded. Shape: (1496, 7)


In [57]:
print("\n📊 Summary statistics:")
display(df.info())


📊 Summary statistics:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1496 entries, 0 to 1495
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1496 non-null   int64  
 1   sex       1496 non-null   object 
 2   bmi       1481 non-null   float64
 3   children  1492 non-null   float64
 4   smoker    1467 non-null   object 
 5   region    1479 non-null   object 
 6   charges   1496 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 81.9+ KB


None

## A quick cleaning

In [59]:
print(df['sex'].unique())
print(df['children'].unique())
print(df['smoker'].unique())
print(df['region'].unique())    

['female' 'male' '35' '61' '11111' '52']
[ 0.  1.  3.  2.  5.  4. 35. 61. nan]
['yes' 'no' nan '61' '?>!' '??' '35']
['southwest' 'southeast' 'northwest' 'northeast' nan '??' '35' '61' '?>!']


In [60]:
valid_sex = ['male', 'female']

df.loc[~df['sex'].isin(valid_sex), 'sex'] = np.nan

In [61]:
valid_smoker = ['yes', 'no']

df.loc[~df['smoker'].isin(valid_smoker), 'smoker'] = np.nan

In [62]:
valid_regions = [
    'southwest',
    'southeast',
    'northwest',
    'northeast'
]

df.loc[~df['region'].isin(valid_regions), 'region'] = np.nan

In [63]:
df["children"] = df["children"].where(df["children"].between(0, 5), np.nan)

In [64]:
print(df['sex'].unique())
print(df['children'].unique())
print(df['smoker'].unique())
print(df['region'].unique())

['female' 'male' nan]
[ 0.  1.  3.  2.  5.  4. nan]
['yes' 'no' nan]
['southwest' 'southeast' 'northwest' 'northeast' nan]


In [65]:
# -------------------------------
# Step 2: Data Cleaning
# -------------------------------

# Drop null values (very small ratio <= 1%)
df.dropna(inplace=True)

# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Post-cleaning check
print("\n✅ After Cleaning:")
print("Remaining nulls:", df.isnull().sum().sum())
print("Remaining duplicates:", df.duplicated().sum())
print("Final shape:", df.shape)


✅ After Cleaning:
Remaining nulls: 0
Remaining duplicates: 0
Final shape: (1252, 7)


# Encoding

In [66]:
print(df['sex'].unique())
print(df['children'].unique())
print(df['smoker'].unique())
print(df['region'].unique())

['female' 'male']
[0. 1. 3. 2. 5. 4.]
['yes' 'no']
['southwest' 'southeast' 'northwest' 'northeast']


## 1. sex

In [67]:
# Convert 'male' to 1 and 'female' to 0
df['sex'] = df['sex'].map({'male': 1, 'female': 0})

# Verify the transformation on your data types
print(df['sex'].value_counts())
print(df['sex'].dtypes)

sex
1    628
0    624
Name: count, dtype: int64
int64


## 2. region

In [68]:
from sklearn.preprocessing import OneHotEncoder
# Clean, one-hot encode the region column for a small dataset
df = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)

# Verify the new columns (you will see exactly 3 region columns instead of 4)
print(df.columns)

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges',
       'region_northwest', 'region_southeast', 'region_southwest'],
      dtype='object')


## 3. smoker

In [69]:
# 1. Map smoker data from the smoker column
df['smoker'] = df['smoker'].map({'yes': 1, 'no': 0})

# 2. Verify the transformation
print(df['smoker'].value_counts())
print(df['smoker'].dtypes)

smoker
0    1002
1     250
Name: count, dtype: int64
int64


In [70]:
df

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0.0,1,16884.92400,0,0,1
1,18,1,33.770,1.0,0,1725.55230,0,1,0
2,28,1,33.000,3.0,0,4449.46200,0,1,0
3,33,1,22.705,0.0,0,21984.47061,1,0,0
4,32,1,28.880,0.0,0,3866.85520,1,0,0
...,...,...,...,...,...,...,...,...,...
1427,53,1,36.600,3.0,0,11264.54100,0,0,1
1452,49,0,30.780,1.0,0,9778.34720,0,0,0
1458,34,0,33.250,1.0,0,5594.84550,0,0,0
1481,52,0,37.525,2.0,0,33471.97189,1,0,0


# Scaling and Checking models

In [71]:
### These all the possible ml we may use
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_validate ,GridSearchCV
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
import timeit
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score

In [72]:
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'charges',
       'region_northwest', 'region_southeast', 'region_southwest'],
      dtype='object')

In [73]:
x , y = df.drop(['charges'] , axis = 1 ) , df['charges']

x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.1 , random_state=7)

In [74]:
models = []
models.append(('LR', LinearRegression()))
models.append(('KNN', KNeighborsRegressor()))
models.append(('RF', RandomForestRegressor()))
models.append(("XG", XGBRegressor()))


In [75]:
scalers= list()
scalers.append(("MinMaxScaler",MinMaxScaler()))
# scalers.append(("StandardScaler",StandardScaler()))
# scalers.append(("RobustScaler",RobustScaler()))

In [76]:
for scaler in scalers:
    print(scaler[0])
    print("=" * 50)
    for model in models:   
        start = timeit.default_timer()
        steps = list()
        steps.append(scaler)
        steps.append(model)
        pipeline = Pipeline(steps=steps)
        
        try:  
            # Added neg_mean_absolute_error to the list
            scores = cross_validate(
                pipeline, x, y, cv=5, 
                scoring=["r2", "neg_mean_squared_error", "neg_mean_absolute_error"], 
                return_train_score=True
            )
        except Exception as e:
            print(f"Skipping {model[0]} due to error: {e}")
            continue
            
        print(model[0])
        print("-" * 25)
        
        # 1. R2 Scores
        print('Train R2 Score  :', scores["train_r2"].mean())
        print('Test R2 Score   :', scores["test_r2"].mean())
        print("-" * 25)
        
        # Extract raw positive MSE values for calculations
        train_mse = -scores["train_neg_mean_squared_error"].mean()
        test_mse = -scores["test_neg_mean_squared_error"].mean()

        
        # 2. Root Mean Squared Error (RMSE) -> Calculated as square root of MSE
        print('Train RMSE      :', train_mse ** 0.5)
        print('Test RMSE       :', test_mse ** 0.5)
        print("-" * 25)
        
        
        stop = timeit.default_timer()
        print("Run Time        :", stop - start)
        print("=" * 50)

MinMaxScaler
LR
-------------------------
Train R2 Score  : 0.7523153543774358
Test R2 Score   : 0.7475926555098568
-------------------------
Train RMSE      : 6013.991362518461
Test RMSE       : 6053.290794309516
-------------------------
Run Time        : 0.20645639998838305
KNN
-------------------------
Train R2 Score  : 0.8483556239992904
Test R2 Score   : 0.7589444380590724
-------------------------
Train RMSE      : 4705.55913305209
Test RMSE       : 5910.330832257683
-------------------------
Run Time        : 0.2637674999423325
RF
-------------------------
Train R2 Score  : 0.9755306601400516
Test R2 Score   : 0.8301861085570387
-------------------------
Train RMSE      : 1889.9460281337538
Test RMSE       : 4947.655262477074
-------------------------
Run Time        : 2.61955209961161
XG
-------------------------
Train R2 Score  : 0.9959142997481287
Test R2 Score   : 0.80308431592845
-------------------------
Train RMSE      : 771.4681848772034
Test RMSE       : 5326.395475414

In [77]:
scalers= list()
#scalers.append(("MinMaxScaler",MinMaxScaler()))
scalers.append(("StandardScaler",StandardScaler()))
# scalers.append(("RobustScaler",RobustScaler()))

In [78]:
for scaler in scalers:
    print(scaler[0])
    print("=" * 50)
    for model in models:   
        start = timeit.default_timer()
        steps = list()
        steps.append(scaler)
        steps.append(model)
        pipeline = Pipeline(steps=steps)
        
        try:  
            # Added neg_mean_absolute_error to the list
            scores = cross_validate(
                pipeline, x, y, cv=5, 
                scoring=["r2", "neg_mean_squared_error", "neg_mean_absolute_error"], 
                return_train_score=True
            )
        except Exception as e:
            print(f"Skipping {model[0]} due to error: {e}")
            continue
            
        print(model[0])
        print("-" * 25)
        
        # 1. R2 Scores
        print('Train R2 Score  :', scores["train_r2"].mean())
        print('Test R2 Score   :', scores["test_r2"].mean())
        print("-" * 25)
        
        # Extract raw positive MSE values for calculations
        train_mse = -scores["train_neg_mean_squared_error"].mean()
        test_mse = -scores["test_neg_mean_squared_error"].mean()

        
        # 2. Root Mean Squared Error (RMSE) -> Calculated as square root of MSE
        print('Train RMSE      :', train_mse ** 0.5)
        print('Test RMSE       :', test_mse ** 0.5)
        print("-" * 25)
        
        
        stop = timeit.default_timer()
        print("Run Time        :", stop - start)
        print("=" * 50)

StandardScaler
LR
-------------------------
Train R2 Score  : 0.7523153543774357
Test R2 Score   : 0.7475926555098568
-------------------------
Train RMSE      : 6013.991362518461
Test RMSE       : 6053.290794309515
-------------------------
Run Time        : 0.12972800014540553
KNN
-------------------------
Train R2 Score  : 0.8621410893350767
Test R2 Score   : 0.7891213277727145
-------------------------
Train RMSE      : 4485.58201849307
Test RMSE       : 5522.039231273237
-------------------------
Run Time        : 0.12295919982716441
RF
-------------------------
Train R2 Score  : 0.9761681194690857
Test R2 Score   : 0.8304747071729686
-------------------------
Train RMSE      : 1864.9808723631957
Test RMSE       : 4939.418744158062
-------------------------
Run Time        : 2.5227939998731017
XG
-------------------------
Train R2 Score  : 0.9959142997481287
Test R2 Score   : 0.80308431592845
-------------------------
Train RMSE      : 771.4681848772034
Test RMSE       : 5326.3954

In [79]:
scalers= list()
#scalers.append(("MinMaxScaler",MinMaxScaler()))
#scalers.append(("StandardScaler",StandardScaler()))
scalers.append(("RobustScaler",RobustScaler()))

In [80]:
for scaler in scalers:
    print(scaler[0])
    print("=" * 50)
    for model in models:   
        start = timeit.default_timer()
        steps = list()
        steps.append(scaler)
        steps.append(model)
        pipeline = Pipeline(steps=steps)
        
        try:  
            # Added neg_mean_absolute_error to the list
            scores = cross_validate(
                pipeline, x, y, cv=5, 
                scoring=["r2", "neg_mean_squared_error", "neg_mean_absolute_error"], 
                return_train_score=True
            )
        except Exception as e:
            print(f"Skipping {model[0]} due to error: {e}")
            continue
            
        print(model[0])
        print("-" * 25)
        
        # 1. R2 Scores
        print('Train R2 Score  :', scores["train_r2"].mean())
        print('Test R2 Score   :', scores["test_r2"].mean())
        print("-" * 25)
        
        # Extract raw positive MSE values for calculations
        train_mse = -scores["train_neg_mean_squared_error"].mean()
        test_mse = -scores["test_neg_mean_squared_error"].mean()

        
        # 2. Root Mean Squared Error (RMSE) -> Calculated as square root of MSE
        print('Train RMSE      :', train_mse ** 0.5)
        print('Test RMSE       :', test_mse ** 0.5)
        print("-" * 25)
        
        
        stop = timeit.default_timer()
        print("Run Time        :", stop - start)
        print("=" * 50)

RobustScaler
LR
-------------------------
Train R2 Score  : 0.7523153543774358
Test R2 Score   : 0.7475926555098568
-------------------------
Train RMSE      : 6013.991362518461
Test RMSE       : 6053.290794309516
-------------------------
Run Time        : 0.22394719999283552
KNN
-------------------------
Train R2 Score  : 0.8517718606460946
Test R2 Score   : 0.7610078546349657
-------------------------
Train RMSE      : 4652.005182867618
Test RMSE       : 5893.126814546534
-------------------------
Run Time        : 0.26123160030692816
RF
-------------------------
Train R2 Score  : 0.9761436744427483
Test R2 Score   : 0.8292859773509796
-------------------------
Train RMSE      : 1866.360926117727
Test RMSE       : 4958.665883872345
-------------------------
Run Time        : 2.2773308004252613
XG
-------------------------
Train R2 Score  : 0.9959142997481287
Test R2 Score   : 0.80308431592845
-------------------------
Train RMSE      : 771.4681848772034
Test RMSE       : 5326.395475

## Model & Scaler Selection Analysis
After evaluating our dataset across three different preprocessing techniques (MinMaxScaler, StandardScaler, and RobustScaler) against our regression models, we observed the following key insights:

- Tree-Based Dominance: Random Forest (RF) and XGBoost (XG) universally outperformed linear and instance-based models across all scaling methodologies.
- The best model is Random Forest (RF) with a Test $R^2$ score of 0.82 and a Test RMSE of 4971.
- XG is overfitting Train R2 Score.

In [47]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from imblearn.pipeline import Pipeline

# 1. Create a clean, single pipeline for The winner
tuning_pipeline = Pipeline(steps=[
    ('scaler', MinMaxScaler()),
    ('rf', RandomForestRegressor(random_state=7))
])

# 2. Define the grid of parameters I  want to test
param_grid = {
    'rf__n_estimators': [100, 150, 200],
    'rf__max_depth': [8, 10, 12, None],          
    'rf__min_samples_split': [2, 5, 10],         
    'rf__min_samples_leaf': [1, 2, 4]            
}

# 3. Set up the Grid Search using your chosen CV strategy
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    return_train_score=True,
    n_jobs=-1 
)

# 4. Fit the grid search on your whole x and y data
print("Tuning your Random Forest model... Please wait...")
grid_search.fit(x, y)

# 5. Display the best results!
print("\n" + "="*40)
print("🏆 BEST PARAMETERS FOUND:")
print("="*40)
for param, value in grid_search.best_params_.items():
    print(f"{param.replace('rf__', '')}: {value}")

print("-"*40)
print(f"Best Test R2 Score achieved: {grid_search.best_score_:.4f}")
print("="*40)

Tuning your Random Forest model... Please wait...

🏆 BEST PARAMETERS FOUND:
max_depth: 8
min_samples_leaf: 4
min_samples_split: 10
n_estimators: 200
----------------------------------------
Best Test R2 Score achieved: 0.8508


### The Final production is locked and ready

## let's check it 

In [48]:
final_model_pipeline = grid_search.best_estimator_

In [49]:
from sklearn.model_selection import cross_validate

# Evaluate the final tuned pipeline
final_scores = cross_validate(
    final_model_pipeline, x, y, cv=5, 
    scoring=["r2", "neg_mean_squared_error", "neg_mean_absolute_error"],
    return_train_score=True
)

print("=========================================")
print("🏆 FINAL TUNED RANDOM FOREST PERFORMANCE")
print("=========================================")
print(f"Train R2 Score : {final_scores['train_r2'].mean():.4f}")
print(f"Test R2 Score  : {final_scores['test_r2'].mean():.4f}  <-- Look at that stability!")
print("-" * 41)

final_test_mse = -final_scores["test_neg_mean_squared_error"].mean()

print(f"Final Test RMSE : ${final_test_mse ** 0.5:,.2f}")
print("=========================================")

🏆 FINAL TUNED RANDOM FOREST PERFORMANCE
Train R2 Score : 0.9060
Test R2 Score  : 0.8508  <-- Look at that stability!
-----------------------------------------
Final Test RMSE : $4,636.55


In [50]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Get feature importances from the 'rf' step inside the final pipeline
importances = final_model_pipeline.named_steps['rf'].feature_importances_

# 2. Match them with your feature column names (assuming x is a pandas DataFrame)
feature_imp_df = pd.DataFrame({
    'Feature': x.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n📊 FEATURE IMPORTANCE RANKING:")
print(feature_imp_df.to_string(index=False))


📊 FEATURE IMPORTANCE RANKING:
         Feature  Importance
          smoker    0.673471
             bmi    0.181836
             age    0.126861
        children    0.010357
             sex    0.002242
region_southeast    0.002105
region_northwest    0.001565
region_southwest    0.001563


### Hyperparameter Tuning Conclusion

Through systematic 5-fold grid search cross-validation, we successfully optimized our **MinMaxScaler + Random Forest** pipeline. 

By applying constraints (`max_depth=8`, `min_samples_leaf=4`, `min_samples_split=10`), we curbed the model's baseline overfitting behavior. As a result, the cross-validated **Test $R^2$ score improved from 0.82 to 0.85**, yielding our most robust and generalizable model for predicting insurance `charges`, and the test score of RMSE has been changed from $4و961 to $4,636.

# Model Check

In [51]:
# 1.    
final_insurance_model = grid_search.best_estimator_

# 2. 
print("🏆 Final Model is officially locked and loaded!")

🏆 Final Model is officially locked and loaded!


In [52]:
dummy_patient = pd.DataFrame([{
    'age': 35,
    'sex': 1,               # male = 1 
    'bmi': 28.5,
    'children': 2,
    'smoker': 0,            # no smoker
    'region_northwest': 0,
    'region_southeast': 1,
    'region_southwest': 0
}])

# let's predict the charges
predicted_charge = final_insurance_model.predict(dummy_patient)
print(f"🔮 Predicted Insurance Charges for Dummy Patient: ${predicted_charge[0]:,.2f}")

🔮 Predicted Insurance Charges for Dummy Patient: $6,579.84


## let's save the file 

In [53]:
import os
import joblib

# 1. Define project folder path
project_folder = r"D:\Data science\Final project\Insurance"
os.makedirs(project_folder, exist_ok=True)

# 2. Extract column schema for your 8 features
input_features = x.columns.tolist()  # ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']

# 3. Save Trained Model
model_path = os.path.join(project_folder, "insurance_rf_model.pkl")
joblib.dump(final_insurance_model, model_path)
print(f"💾 1. Model saved to: {model_path}")

# 4. Save Feature Order (using .h5 as requested)
input_h5_path = os.path.join(project_folder, "input.h5")
joblib.dump(input_features, input_h5_path)
print(f"📋 2. Input features saved to: {input_h5_path}")

# 5. Save Metadata Dict (for Streamlit app compatibility)
metadata_path = os.path.join(project_folder, "model_metadata.joblib")
joblib.dump({"features": input_features}, metadata_path)
print(f"⚙️ 3. Model metadata saved to: {metadata_path}")

💾 1. Model saved to: D:\Data science\Final project\Insurance\insurance_rf_model.pkl
📋 2. Input features saved to: D:\Data science\Final project\Insurance\input.h5
⚙️ 3. Model metadata saved to: D:\Data science\Final project\Insurance\model_metadata.joblib


# The cleand Csv File

In [54]:
# 6. Save Cleaned Dataset (for Streamlit EDA/Data tabs)
csv_path = os.path.join(project_folder, "insurance_cleaned.csv")
df.to_csv(csv_path, index=False)
print(f"📊 4. Cleaned dataset saved to: {csv_path}\n")

print("🎉 All 4 deployment assets are locked and ready in your Insurance project directory!")

📊 4. Cleaned dataset saved to: D:\Data science\Final project\Insurance\insurance_cleaned.csv

🎉 All 4 deployment assets are locked and ready in your Insurance project directory!
